# LORNet training (CFA-SeeSR V2): learned observation reliability on Google Colab (T4)

LORNet is a ~33 k-parameter convolutional network that predicts, from the conditioning image (the upscaled LR),
where the LR observation agrees with the HR image in the frozen SeeSR VAE's latent space. It is the **only**
trainable part of CFA-SeeSR; the SeeSR UNet, ControlNet, DAPE and VAE stay frozen, and only the VAE is loaded here
(once, to cache latent targets). Independent of `seesr_evaluation.ipynb`, which only reads the checkpoint.

Pipeline (details: `experiments/training/README.md`, diagram D in `docs/ARCHITECTURE.md`):
1. pair the RealSR / DRealSR **training** images; hold out a hash-selected validation split; refuse test folders
   and any image stem that also appears in the evaluation folders;
2. cache: conditioning image (upstream resizing) and HR crop → frozen VAE (posterior mean × scaling factor) →
   E = LocalAverage(|LPF(z_lr − z_hr)|); τ = median(E_train)/ln 2; target R = exp(−E/τ);
3. train LORNet (Charbonnier + λ_TV·TV, AdamW, cosine, early stopping on validation loss, fixed seed);
4. evaluate on the validation split against the teacher, next to the heuristic map;
5. write the V2 screening images (validation split) used by `seesr_evaluation.ipynb` section 17.

The teacher is a **proxy** for observation fidelity; whether correcting toward the LR latent improves the output is
tested separately (map-validity test and the ablation).

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv

## 2. Configuration (edit this cell only)

In [ ]:
import os, json, pathlib, subprocess
REPO_URL   = "https://github.com/rithvik318/SeeSR.git"
BRANCH     = "research/cfa-seesr-v2"
REPO_DIR   = "/content/SeeSR"
DRIVE_ROOT = "/content/drive/MyDrive/seesr"
MODELS_DIR = f"{DRIVE_ROOT}/models"                  # needs stable-diffusion-2-base/vae only
DATA_DIR   = f"{DRIVE_ROOT}/datasets"
ENV_DIR    = "/content/seesr-env"
PY         = f"{ENV_DIR}/bin/python"
LORNET_OUT = f"{DRIVE_ROOT}/lornet"                  # cache/, checkpoints/best.pt (read by seesr_evaluation.ipynb)
V2_SCREEN_DATA = f"{DRIVE_ROOT}/v2_screening_data"   # validation images for the V2 screening
# TRAINING splits only (RealSR V3: HR and LR4 in one folder per camera; DRealSR x4: train_HR / train_LR)
TRAIN_SOURCES = [
    {"name": "realsr_canon", "group": "RealSR", "hr_dir": f"{DATA_DIR}/RealSR_V3/Canon/Train/4",
     "lr_dir": f"{DATA_DIR}/RealSR_V3/Canon/Train/4", "hr_pattern": "*_HR.png", "lr_pattern": "*_LR4.png"},
    {"name": "realsr_nikon", "group": "RealSR", "hr_dir": f"{DATA_DIR}/RealSR_V3/Nikon/Train/4",
     "lr_dir": f"{DATA_DIR}/RealSR_V3/Nikon/Train/4", "hr_pattern": "*_HR.png", "lr_pattern": "*_LR4.png"},
    {"name": "drealsr", "group": "DRealSR", "hr_dir": f"{DATA_DIR}/DRealSR/x4/Train_x4/train_HR",
     "lr_dir": f"{DATA_DIR}/DRealSR/x4/Train_x4/train_LR"},
]
# evaluation LR folders (the test sets of seesr_evaluation.ipynb): any shared image stem stops the cache step
EXCLUDE_DIRS = [f"{DATA_DIR}/RealSR/test_LR", f"{DATA_DIR}/DRealSR/test_LR"]
SEED = 2026
EPOCHS, BATCH = 60, 16             # T4: ~1-2 min per epoch for ~2k crops (estimate, not measured)

os.environ.update(PYTHONDONTWRITEBYTECODE="1")

def sh(cmd):
    proc = subprocess.Popen(["bash", "-c", cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait():
        raise RuntimeError(f"exit code {proc.returncode}: {cmd[:300]}")

## 3. Drive, code and environment (same pinned stack as the evaluation notebook)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
if not os.path.isdir(REPO_DIR):
    sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
sh("git log -1 --format='%H %s' && git status --short")
sh("pip -q install uv && uv python install 3.10")
if not os.path.exists(PY):
    sh(f"uv venv --seed --python 3.10 {ENV_DIR}")
sh(f"{PY} -m pip -q install -r experiments/requirements-colab.txt pytest")
if not os.path.exists("preset/models"):
    os.makedirs("preset", exist_ok=True); os.symlink(MODELS_DIR, "preset/models")
sh(f"{PY} -m pytest -q -p no:cacheprovider experiments/tests/test_lornet.py")

## 4. Training configuration

Copies `experiments/training/configs/lornet.yaml` with the paths above to `LORNET_OUT/config.yaml` (recorded with
the checkpoint). Change the model, loss or schedule there or with `--set`.

In [ ]:
import yaml
cfg = yaml.safe_load(open("experiments/training/configs/lornet.yaml"))
cfg["seed"] = SEED
cfg["output_dir"] = LORNET_OUT
cfg["vae"]["pretrained_model_path"] = f"{MODELS_DIR}/stable-diffusion-2-base"
cfg["data"]["sources"] = TRAIN_SOURCES
cfg["data"]["exclude_dirs"] = [d for d in EXCLUDE_DIRS if os.path.isdir(d)]
cfg["train"].update(epochs=EPOCHS, batch_size=BATCH)
os.makedirs(LORNET_OUT, exist_ok=True)
CFG = f"{LORNET_OUT}/config.yaml"
yaml.safe_dump(cfg, open(CFG, "w"), sort_keys=False)
missing = [s for s in TRAIN_SOURCES if not os.path.isdir(s["hr_dir"])]
print("config:", CFG, "| missing sources:", [s["name"] for s in missing] or "none")
print("excluded evaluation folders:", cfg["data"]["exclude_dirs"])

## 5. Latent cache (frozen VAE, fp16; once)

In [ ]:
if not pathlib.Path(LORNET_OUT, "cache", "manifest.json").exists():
    sh(f"{PY} experiments/training/train_lornet.py cache --config {CFG} 2>&1 | tee {LORNET_OUT}/cache.log")
print(json.dumps(json.loads(pathlib.Path(LORNET_OUT, "cache", "manifest.json").read_text())["splits"], indent=1))

## 6. Train (early stopping on validation loss; best.pt = lowest validation loss)

In [ ]:
sh(f"{PY} experiments/training/train_lornet.py train --config {CFG} 2>&1 | tee {LORNET_OUT}/train.log")
print(pathlib.Path(LORNET_OUT, "train_summary.json").read_text())

## 7. Evaluate on the validation split

LORNet vs the teacher (MAE, RMSE, Pearson, Spearman, ECE) and, rank metrics only, the heuristic map on the same
crops; parameter count, MACs and inference time at 512² on this GPU.

In [ ]:
sh(f"{PY} experiments/training/train_lornet.py evaluate --config {CFG}")

## 8. V2 screening images (validation split; used by seesr_evaluation.ipynb section 17)

In [ ]:
sh(f"{PY} experiments/training/make_screening_set.py --config {CFG} --n 20 --crop_hr 512 --output {V2_SCREEN_DATA}")
print("checkpoint for seesr_evaluation.ipynb:", f"{LORNET_OUT}/checkpoints/best.pt")
from google.colab import drive as _d; _d.flush_and_unmount()